# Chapter 3: RAG and Retrieval Evaluation

Estimated time: about 8 hours (the longest chapter in this course), including a 15-20 minute
skippable concept refresher near the start.

Prerequisites: Chapters 1-2 (`agentlib.llm_client`, the real/mock brain toggle).

Interview category this chapter maps to: RAG evaluation and diagnosis, such as "is this a
retrieval problem or a generation problem, and which metric proves it," chunking strategy for
messy real-world corpora, and explaining hallucination-despite-grounding to a non-technical
stakeholder.


## Quick review (skippable)

If your background already covers information retrieval or search/recsys, skip this. It's a
safety net, not an assumption of a gap. Three concepts this chapter leans on:

TF-IDF weights a word by how often it appears in this document (term frequency) against how
rare it is across all documents (inverse document frequency). A word that's common everywhere
(like "the") gets down-weighted; a word that's rare overall but frequent in one document is
probably what that document is actually about. It's the simplest way to turn text into
comparable vectors, and it has no notion of meaning: "car" and "automobile" are completely
unrelated to a TF-IDF vectorizer.

Embeddings work differently: an embedding model maps text to a vector such that distance in
vector space corresponds to similarity in meaning. "Car" and "automobile" end up close
together even though they share zero characters, because the model learned that they're used
in similar contexts. This is what lets embedding-based retrieval find a relevant passage that
doesn't share any of the query's exact words, which TF-IDF structurally cannot do.

MRR, or mean reciprocal rank, is the third concept worth knowing cold. For a single query,
take 1 divided by the rank of the first correct result (rank 1 -> 1.0, rank 2 -> 0.5, rank 5
-> 0.2, never found -> 0). Average that across all your queries and you get MRR: a standard
information-retrieval metric, but a more specialized one than precision/recall, so it's easy
to have missed it outside a dedicated search/recsys course.


## Concept: three separate failure surfaces

A RAG pipeline has three stages that fail in different ways, and diagnosing which one failed
is one of the most common real interview questions in this space:

| Stage | Question it answers | Fails when |
|---|---|---|
| Retrieval | Did we find the right document(s)? | The right passage exists in the corpus but wasn't returned in the top-k |
| Ranking | Among what we found, is the best one near the top? | The right passage was retrieved but buried below less-relevant ones |
| Generation | Did the model actually use what was retrieved? | The right passage was retrieved and ranked well, but the model ignored it anyway |

Each has a different metric that isolates it. Precision@k and recall@k diagnose retrieval
(did we get the right documents at all), MRR diagnoses ranking (how high up is the first
correct one), and a faithfulness or groundedness check diagnoses generation (does the
answer's content actually trace back to what was retrieved, rather than the model's own
parametric knowledge). This chapter's break-it section is built around forcing each of these
to fail independently, so the distinction stops being abstract.

Why doesn't RAG eliminate hallucination? Retrieval only guarantees relevant material was
placed in front of the model; nothing forces the model to actually use it faithfully. This
chapter demonstrates that gap directly, not just asserts it.


## Setup

In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import random
import re

import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from agentlib import eval_metrics, llm_client, synthetic_data
from agentlib.grading import check

random.seed(42)
np.random.seed(42)

nlp = spacy.load("en_core_web_md")

print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")


LLM_PROVIDER = 'anthropic', HAS_KEY = False


## Build: a real corpus, not hand-written Q&A

The base corpus is 25-30 real passages from SQuAD 1.1 (Rajpurkar et al., 2016; see
`REFERENCES.md`), which already ships human-annotated ground-truth question/answer pairs
mapped to each passage, so we use those directly instead of hand-writing synthetic Q&A.
License: CC BY-SA 4.0, inherited from the underlying Wikipedia content.

`agentlib.synthetic_data.load_squad_sample()` loads this from a committed local cache
(`data/rag_corpus/squad_sample.json`) rather than fetching it live. See that module's
docstring and `PROGRESS.md`'s Unit 4 notes for exactly why (short version: this repo's build
environment can't reach Hugging Face or SEC EDGAR, so the real SQuAD data here was fetched
once, from its canonical GitHub source, and cached; no notebook run, including CI, needs
network access for it).


In [2]:
squad = synthetic_data.load_squad_sample()
docs = squad["docs"]
qa_pairs = squad["qa_pairs"]

print(f"Loaded {len(docs)} real SQuAD passages and {len(qa_pairs)} real QA pairs.")
print(f"Source: {squad['source']}")
print(f"License: {squad['license']}")
print()
print("Example passage:", docs[0]["title"])
print(" ", docs[0]["text"][:200], "...")
print("Example QA:", qa_pairs[0]["question"], "->", qa_pairs[0]["answer"])


Loaded 28 real SQuAD passages and 55 real QA pairs.
Source: SQuAD 1.1 dev set (Rajpurkar et al., 2016), official rajpurkar/SQuAD-explorer GitHub repo
License: CC BY-SA 4.0 (inherited from underlying Wikipedia content; see REFERENCES.md)

Example passage: Genghis_Khan
  The Shah's army was split by diverse internecine feuds and by the Shah's decision to divide his army into small groups concentrated in various cities. This fragmentation was decisive in Khwarezmia's d ...
Example QA: What feature of the Shah's army enable the weary Mongol forces easy early victories? -> fragmentation


4-6 near-duplicate "confusable" documents, layered synthetically on top of the real corpus
(real SQuAD passages won't reliably contain the specific near-duplicate pairs the break-it
section needs). Each is a real passage with exactly one fact swapped, similar enough to be
retrieved in place of the original, wrong if you check the swapped fact. Deterministic, seeded,
built in `agentlib.synthetic_data.generate_confusable_documents()`.

In [3]:
confusables = synthetic_data.generate_confusable_documents(docs, n=5, seed=42)
all_docs = docs + confusables

print(f"Generated {len(confusables)} confusable documents. Corpus is now {len(all_docs)} docs.")
for c in confusables[:2]:
    original = next(d for d in docs if d["doc_id"] == c["confusable_of"])
    print(f"\n{c['confusable_of']} (original):   {original['text'][:120]}...")
    print(f"{c['doc_id']} (confusable): {c['text'][:120]}...")
    print(f"  synthetic change: {c['synthetic_change']}")


Generated 5 confusable documents. Corpus is now 33 docs.

squad-027 (original):   In anglophone academic works, theories regarding imperialism are often based on the British experience. The term "Imperi...
squad-027-confusable (confusable): In anglophone academic works, theories regarding imperialism are often based on the British experience. The term "Imperi...
  synthetic change: name 'Minister Benjamin' -> 'Sarah Thompson'

squad-003 (original):   Genghis Khan is regarded as one of the prominent leaders in Mongolia's history. He is responsible for the emergence of t...
squad-003-confusable (confusable): Sarah Thompson is regarded as one of the prominent leaders in Mongolia's history. He is responsible for the emergence of...
  synthetic change: name 'Genghis Khan' -> 'Sarah Thompson'


## Build: ingesting a genuinely messy real document

Retrieval quality is only as good as what gets indexed. Real-world documents are rarely
clean paragraphs. They're long, unevenly structured, full of embedded code and links, and
often contain near-duplicate sections. "How would you chunk a messy real-world corpus?" is a
genuine, commonly-asked interview question, so this uses a genuinely messy real document
rather than faking the mess: `anthropic-sdk-python`'s actual `CHANGELOG.md` (see
`REFERENCES.md` and `PROGRESS.md`'s Unit 4 notes for why this replaced the spec's original
SEC-EDGAR-filings suggestion, since SEC EDGAR is unreachable from this build environment).
It's real, unedited, and has exactly the properties this exercise needs: inconsistent
section lengths, embedded links and commit hashes, and, usefully, many genuinely
near-duplicate release-note sections, which is a good stand-in for the deduplication exercise
specifically.


In [4]:
messy_text = synthetic_data.load_messy_corpus()
print(f"Loaded {len(messy_text)} characters / {len(messy_text.splitlines())} lines of real, unedited changelog text.")
print()
print(messy_text[:600])
print("...")


Loaded 50745 characters / 900 lines of real, unedited changelog text.

# Changelog

## 0.121.0 (2026-08-07)

Full Changelog: [v0.120.2...v0.121.0](https://github.com/anthropics/anthropic-sdk-python/compare/v0.120.2...v0.121.0)

### Features

* **api:** add `mid-conversation-tool-changes-2026-07-01` beta ([c7d1531](https://github.com/anthropics/anthropic-sdk-python/commit/c7d1531d9d63a35430333039f8c975bba4ef0411))
* **api:** add support for session budgets, advisor tool, pinned inference location and skills auto-loading from GitHub ([193bae0](https://github.com/anthropics/anthropic-sdk-python/commit/193bae02806219047d484d7efcb9b91776a6c45c))


### Chores

* **api:
...


### Chunking strategy: fixed-size vs. semantic

Fixed-size chunking splits on a raw character or word count, with no regard for document
structure. It's simple and predictable, but it will happily cut a sentence (or a code block)
in half. Semantic, or sentence/section-boundary, chunking splits on natural boundaries
instead (here, on `##`/`###` headers, since this document is Markdown). It's more expensive
to implement, but each chunk is a coherent unit. Worked side by side against the same messy
input:


In [5]:
def chunk_fixed_size(text: str, chunk_size: int = 400) -> list:
    '''Splits on a raw character count -- fast, simple, structure-blind.'''
    return [text[i : i + chunk_size] for i in range(0, len(text), chunk_size)]


def chunk_by_section(text: str) -> list:
    '''Splits on Markdown section headers (## or ###) -- respects document structure, so a
    chunk is never a sentence or code block cut in half.'''
    sections = re.split(r"\n(?=##+\s)", text)
    return [s.strip() for s in sections if s.strip()]


fixed_chunks = chunk_fixed_size(messy_text)
section_chunks = chunk_by_section(messy_text)

print(f"Fixed-size chunking:  {len(fixed_chunks)} chunks")
print(f"Semantic (section) chunking: {len(section_chunks)} chunks")
print()
print("--- Fixed-size chunk #3 (notice it starts/ends mid-sentence) ---")
print(repr(fixed_chunks[3][:250]))
print()
print("--- Semantic chunk #3 (starts cleanly at a section header) ---")
print(repr(section_chunks[3][:250]))


Fixed-size chunking:  127 chunks
Semantic (section) chunking: 188 chunks

--- Fixed-size chunk #3 (notice it starts/ends mid-sentence) ---
'[d52999b](https://github.com/anthropics/anthropic-sdk-python/commit/d52999b4f66002ffe8263756c59ab0e276c74264))\n\n## 0.120.2 (2026-07-28)\n\nFull Changelog: [v0.120.1...v0.120.2](https://github.com/anthropics/anthropic-sdk-python/compare/v0.120.1...v0.12'

--- Semantic chunk #3 (starts cleanly at a section header) ---
'### Chores\n\n* **api:** remove retired Claude Opus 4.1 models ([5352a33](https://github.com/anthropics/anthropic-sdk-python/commit/5352a33ade5b62ad782b522f1791b7eeeeff83be))\n* **docs:** small updates to descriptions ([b8d4176](https://github.com/anthr'


### Deduplicating near-identical chunks

This changelog has many release sections that are near-identical in structure: the same kind
of "Bug Fixes" bullet, referencing a commit hash and link, repeated across dozens of
releases. Indexing every one of those as a separate chunk wastes index space and can crowd
out more distinctive chunks in retrieval, so we catch them before they enter the index.


In [6]:
def deduplicate_chunks(chunks: list, threshold: float = 0.85) -> list:
    '''Drops any chunk that's near-identical (TF-IDF cosine similarity above threshold) to a
    chunk already kept -- a real, working near-duplicate filter, not just a description of
    one.'''
    if not chunks:
        return []
    vectorizer = TfidfVectorizer().fit(chunks)
    vectors = vectorizer.transform(chunks)

    kept_indices = [0]
    for i in range(1, len(chunks)):
        sims = cosine_similarity(vectors[i], vectors[kept_indices])[0]
        if sims.max() < threshold:
            kept_indices.append(i)
    return [chunks[i] for i in kept_indices]


deduped_chunks = deduplicate_chunks(section_chunks)
print(f"Before dedup: {len(section_chunks)} section chunks")
print(f"After dedup:  {len(deduped_chunks)} chunks ({len(section_chunks) - len(deduped_chunks)} near-duplicates removed)")


Before dedup: 188 section chunks
After dedup:  184 chunks (4 near-duplicates removed)


## Build: two retrieval implementations, compared on the real corpus

TF-IDF (via scikit-learn, no download needed) and local embeddings, using spaCy's
`en_core_web_md` here rather than the spec's originally-suggested `sentence-transformers`,
since the latter needs a Hugging Face model download this build environment can't reach.
spaCy's model installs directly from `requirements.txt` with no separate download step for
anyone. Both retrievers implement the same `retrieve(query, k)` interface so they're directly
comparable.


In [7]:
class TfidfRetriever:
    def __init__(self, docs: list):
        self.doc_ids = [d["doc_id"] for d in docs]
        self.vectorizer = TfidfVectorizer()
        self.doc_vectors = self.vectorizer.fit_transform([d["text"] for d in docs])

    def retrieve(self, query: str, k: int = 3) -> list:
        query_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(query_vec, self.doc_vectors)[0]
        ranked = sims.argsort()[::-1][:k]
        return [self.doc_ids[i] for i in ranked]


class EmbeddingRetriever:
    def __init__(self, docs: list, nlp):
        self.doc_ids = [d["doc_id"] for d in docs]
        self.nlp = nlp
        self.doc_vectors = np.array([nlp(d["text"]).vector for d in docs])

    def retrieve(self, query: str, k: int = 3) -> list:
        query_vec = self.nlp(query).vector
        norms = np.linalg.norm(self.doc_vectors, axis=1) * np.linalg.norm(query_vec) + 1e-9
        sims = (self.doc_vectors @ query_vec) / norms
        ranked = sims.argsort()[::-1][:k]
        return [self.doc_ids[i] for i in ranked]


tfidf_retriever = TfidfRetriever(all_docs)
embedding_retriever = EmbeddingRetriever(all_docs, nlp)

sample_query = qa_pairs[0]["question"]
print("Query:", sample_query)
print("TF-IDF top 3:    ", tfidf_retriever.retrieve(sample_query, k=3))
print("Embedding top 3: ", embedding_retriever.retrieve(sample_query, k=3))
print("Gold doc:        ", qa_pairs[0]["gold_doc_id"])


Query: What feature of the Shah's army enable the weary Mongol forces easy early victories?
TF-IDF top 3:     ['squad-000', 'squad-000-confusable', 'squad-020']
Embedding top 3:  ['squad-000', 'squad-000-confusable', 'squad-015']
Gold doc:         squad-000


### What a real local vector store looks like

Hand-rolled cosine similarity over a NumPy array works fine at this corpus's scale, but a
real system uses a proper vector index. Here's the same corpus indexed with FAISS (Johnson,
Douze & Jégou, 2017; see `REFERENCES.md`), which is what you'd actually reach for in
production instead of the loop above. (Chroma is the other common choice; FAISS was picked
for this course for CI-reliability reasons, see `PROGRESS.md`'s Unit 1 notes.)


In [8]:
import faiss

embedding_matrix = embedding_retriever.doc_vectors.astype("float32")
faiss.normalize_L2(embedding_matrix)  # so inner product == cosine similarity

index = faiss.IndexFlatIP(embedding_matrix.shape[1])
index.add(embedding_matrix)

query_vec = nlp(sample_query).vector.astype("float32").reshape(1, -1)
faiss.normalize_L2(query_vec)
scores, indices = index.search(query_vec, k=3)

faiss_top_3 = [embedding_retriever.doc_ids[i] for i in indices[0]]
print("FAISS top 3:", faiss_top_3)
print("(matches the hand-rolled EmbeddingRetriever above, as it should -- same vectors, same metric)")


FAISS top 3: ['squad-000', 'squad-000-confusable', 'squad-015']
(matches the hand-rolled EmbeddingRetriever above, as it should -- same vectors, same metric)


## Build: the eval harness

Three metrics, and each one answers a different question about the same ranked list:

- **precision@k** — of what you showed the user, how much was worth showing? A retriever
  that pads the top-k with junk is punished here.
- **recall@k** — of what you should have found, how much did you actually find? A retriever
  that returns one perfect document and misses four others is punished here.
- **MRR** — how far down did the user have to read before hitting something useful? Only
  the position of the *first* relevant result matters.

You write all three. `evaluate_retrieval` in `agentlib.eval_metrics` is only the plumbing
that loops over queries and averages; it takes your three functions as arguments, so the
numbers the two retrievers get scored on below are the ones your code produced.

Each cell ends with `your_function = check("task-id", your_function)`. That call runs a
suite of assertions against what you wrote and, if any fail, prints exactly which ones and
why; it hands your function straight back, so the rebinding is a no-op and later cells just
use the name as normal. Every cell from here on that starts with `raise NotImplementedError`
is yours to fill in. Run `python grade.py` at any point to see where you stand.

### precision@k

Watch the denominator. If you ask for `k=5` and the index only has 2 documents, dividing by
`k` scores a retriever badly for something it had no control over.

In [ ]:
def precision_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Of the top-k retrieved documents, what fraction are actually relevant?

    retrieved_ids -- doc_ids in rank order, best first
    relevant_ids  -- the doc_ids that genuinely answer this query
    k             -- how far down the ranking to look

    An empty top-k scores 0.0 rather than raising.
    """
    raise NotImplementedError("Implement me, then re-run this cell")


precision_at_k = check("ch03-precision-k", precision_at_k)

### recall@k

Different denominator, and that difference is the whole distinction between the two metrics.
Recall asks about the documents that *should* have been found, including the ones your
retriever never returned at all.

In [ ]:
def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Of all the documents that were actually relevant, what fraction showed up in the
    top-k?

    A query with no relevant documents scores 0.0 rather than raising.
    """
    raise NotImplementedError("Implement me, then re-run this cell")


recall_at_k = check("ch03-recall-k", recall_at_k)

### Mean reciprocal rank

Reciprocal rank is a per-query score; averaging it across a query set is what makes it
*mean* reciprocal rank, which `evaluate_retrieval` does for you. Ranks here are 1-based: the
top result is rank 1, not rank 0.

In [ ]:
def mean_reciprocal_rank(retrieved_ids: list, relevant_ids: set) -> float:
    """1 / (rank of the first relevant result), or 0.0 if none of the retrieved results are
    relevant. Ranks are 1-based.
    """
    raise NotImplementedError("Implement me, then re-run this cell")


mean_reciprocal_rank = check("ch03-mrr", mean_reciprocal_rank)

With all three passing, run them over every real SQuAD question in the corpus and compare
the two retrievers head to head:

In [9]:
eval_queries = [
    {"query": qa["question"], "relevant_doc_ids": {qa["gold_doc_id"]}}
    for qa in qa_pairs
]

metric_fns = {
    "precision_fn": precision_at_k,
    "recall_fn": recall_at_k,
    "mrr_fn": mean_reciprocal_rank,
}
tfidf_results = eval_metrics.evaluate_retrieval(eval_queries, tfidf_retriever.retrieve, k=3, **metric_fns)
embedding_results = eval_metrics.evaluate_retrieval(eval_queries, embedding_retriever.retrieve, k=3, **metric_fns)

print(f"Evaluated on {len(eval_queries)} real SQuAD questions, k=3\n")
print(f"{'metric':15s} {'TF-IDF':>10s} {'Embeddings':>12s}")
for metric in ["precision@k", "recall@k", "mrr"]:
    print(f"{metric:15s} {tfidf_results[metric]:>10.3f} {embedding_results[metric]:>12.3f}")


Evaluated on 55 real SQuAD questions, k=3

metric              TF-IDF   Embeddings
precision@k          0.315        0.206
recall@k             0.945        0.618
mrr                  0.861        0.536


### Mapping this to RAGAS and DeepEval

The metrics above are hand-built so you understand exactly what they compute. In practice,
you'd reach for a maintained library. RAGAS (Es et al., 2024; see `REFERENCES.md`) computes
context precision/recall analogous to precision@k/recall@k above, plus LLM-judged metrics
like faithfulness (does the answer's content trace back to the retrieved context, a more
sophisticated version of this chapter's `faithfulness_score`) and answer relevancy. DeepEval
covers similar ground with a pytest-style testing interface, useful if you want
retrieval-quality regressions to fail CI the same way a broken unit test would. Both are
worth naming in an interview: "I know the metric definitions and I know RAGAS/DeepEval
implement more sophisticated, often LLM-judged versions of the same ideas at scale" is a
stronger answer than only knowing one layer.


## Build: generation, real by default

Reusing `agentlib.llm_client` and the `HAS_KEY` toggle from Chapter 1 (no new setup here,
just import and check): the generation step calls a real model by default, so you see an
actual model's behavior when it ignores or misuses retrieved context, rather than only a
canned mock version. Falls back to a template-based extractive generator only if no key is
present.

In [10]:
def template_generate(query: str, retrieved_docs: list) -> str:
    '''Deterministic, template-based generator stand-in: picks the best-matching (highest
    query word-overlap) sentence from EACH retrieved doc and concatenates them -- crude, but
    free, explainable, fully offline, and it genuinely uses more of the context when more
    context is retrieved (unlike picking one single best sentence globally, which would
    ignore additional retrieved docs entirely).'''
    query_words = set(re.findall(r"[a-z0-9]+", query.lower()))

    def overlap(sentence):
        return len(query_words & set(re.findall(r"[a-z0-9]+", sentence.lower())))

    best_per_doc = []
    for doc in retrieved_docs:
        sentences = re.split(r"(?<=[.!?])\s+", doc["text"])
        best = max(sentences, key=overlap, default="").strip()
        if best:
            best_per_doc.append(best)
    return " ".join(best_per_doc)


def real_generate(query: str, retrieved_docs: list) -> str:
    context = "\n\n".join(d["text"] for d in retrieved_docs)
    response = llm_client.call_model(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer using ONLY the context above, in one sentence.",
        }],
        model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
    )
    return response.text


generate = real_generate if llm_client.HAS_KEY else template_generate
doc_lookup = {d["doc_id"]: d for d in all_docs}


def rag_answer(query: str, retriever, generate_fn, k: int = 3) -> dict:
    retrieved_ids = retriever.retrieve(query, k)
    retrieved_docs = [doc_lookup[doc_id] for doc_id in retrieved_ids]
    answer = generate_fn(query, retrieved_docs)
    return {"answer": answer, "retrieved_ids": retrieved_ids, "retrieved_docs": retrieved_docs}


demo_qa = qa_pairs[5]
result = rag_answer(demo_qa["question"], tfidf_retriever, generate)
print(f"Using: {'real model' if llm_client.HAS_KEY else 'template_generate (mock, no key present)'}")
print("Question:", demo_qa["question"])
print("Answer:  ", result["answer"])
print("Expected:", demo_qa["answer"])


Using: template_generate (mock, no key present)
Question: By which year did Chrysler ended its full sized luxury model?
Answer:   Chrysler ended production of their full-sized luxury sedans at the end of the 1981 model year, moving instead to a full front-wheel drive lineup for 1982 (except for the M-body Dodge Diplomat/Plymouth Gran Fury and Chrysler New Yorker Fifth Avenue sedans). However, it seems that if they had meant to call Genghis tenggis they could have said, and written, "Tenggis Khan", which they did not.) Zhèng (Chinese: 正) meaning "right", "just", or "true", would have received the Mongolian adjectival modifier -s, creating "Jenggis", which in medieval romanization would be written "Genghis". Most Western countries, and some others, have now banned it, but it remains lawful in the United States following a US Supreme Court decision in 1977 which held that paddling did not violate the US Constitution.
Expected: 1981


## Break it #1: a corrupted retriever for a subset of docs

Simulating an index corruption or a bad deploy affecting some slice of traffic: for a
specific subset of documents, retrieval silently returns the wrong doc instead. Generation
has no way to know retrieval failed, so it produces a confident-sounding answer anyway.

In [11]:
CORRUPTED_DOC_IDS = {qa_pairs[i]["gold_doc_id"] for i in range(3)}
WRONG_REPLACEMENT_ID = all_docs[15]["doc_id"]


class CorruptedRetriever:
    def __init__(self, base_retriever):
        self.base = base_retriever

    def retrieve(self, query: str, k: int = 3) -> list:
        results = self.base.retrieve(query, k)
        return [WRONG_REPLACEMENT_ID if doc_id in CORRUPTED_DOC_IDS else doc_id for doc_id in results]


corrupted_retriever = CorruptedRetriever(tfidf_retriever)

print("--- Bug: corrupted retriever for a subset of docs ---\n")
buggy_eval = eval_metrics.evaluate_retrieval(eval_queries, corrupted_retriever.retrieve, k=3, **metric_fns)
healthy_eval = eval_metrics.evaluate_retrieval(eval_queries, tfidf_retriever.retrieve, k=3, **metric_fns)
print(f"Healthy precision@k/recall@k/mrr:   {healthy_eval['precision@k']:.3f} / {healthy_eval['recall@k']:.3f} / {healthy_eval['mrr']:.3f}")
print(f"Corrupted precision@k/recall@k/mrr: {buggy_eval['precision@k']:.3f} / {buggy_eval['recall@k']:.3f} / {buggy_eval['mrr']:.3f}")

affected_qa = qa_pairs[0]
buggy_result = rag_answer(affected_qa["question"], corrupted_retriever, generate)
print(f"\nExample -- Question: {affected_qa['question']}")
print(f"Retrieved (WRONG): {buggy_result['retrieved_ids']} -- should include {affected_qa['gold_doc_id']!r}")
print(f"Answer (confident but wrong): {buggy_result['answer']}")


--- Bug: corrupted retriever for a subset of docs ---

Healthy precision@k/recall@k/mrr:   0.315 / 0.945 / 0.861
Corrupted precision@k/recall@k/mrr: 0.291 / 0.873 / 0.797

Example -- Question: What feature of the Shah's army enable the weary Mongol forces easy early victories?
Retrieved (WRONG): ['squad-015', 'squad-000-confusable', 'squad-020'] -- should include 'squad-000'
Answer (confident but wrong): The Mongol military was also successful in siege warfare, cutting off resources for cities and towns by diverting certain rivers, taking enemy prisoners and driving them in front of the army, and adopting new ideas, techniques and tools from the people they conquered, particularly in employing Muslim and Chinese siege engines and engineers to aid the Mongol cavalry in capturing cities. Michael Chen's army was split by diverse internecine feuds and by the Shah's decision to divide his army into small groups concentrated in various cities. Later came The Clouded Yellow (1951) and Payroll

In [12]:
print("--- Fix: restore correct retrieval ---\n")
fixed_eval = eval_metrics.evaluate_retrieval(eval_queries, tfidf_retriever.retrieve, k=3, **metric_fns)
print(f"Restored precision@k/recall@k/mrr: {fixed_eval['precision@k']:.3f} / {fixed_eval['recall@k']:.3f} / {fixed_eval['mrr']:.3f}")

fixed_result = rag_answer(affected_qa["question"], tfidf_retriever, generate)
print(f"\nRetrieved (correct): {fixed_result['retrieved_ids']}")
print(f"Answer: {fixed_result['answer']}")
print(f"Expected: {affected_qa['answer']}")


--- Fix: restore correct retrieval ---



Restored precision@k/recall@k/mrr: 0.315 / 0.945 / 0.861

Retrieved (correct): ['squad-000', 'squad-000-confusable', 'squad-020']
Answer: The Shah's army was split by diverse internecine feuds and by the Shah's decision to divide his army into small groups concentrated in various cities. Michael Chen's army was split by diverse internecine feuds and by the Shah's decision to divide his army into small groups concentrated in various cities. Later came The Clouded Yellow (1951) and Payroll (1961), both of which feature more extensive scenes filmed in the city.
Expected: fragmentation


## Break it #2: retrieval succeeds, generation ignores it anyway

The failure this chapter's concept section warned about: nothing stops a generator from
answering from its own parametric knowledge even when the retrieved context is correct.
Precision, recall, and MRR are all fine here; the bug is invisible to retrieval metrics
entirely. Faithfulness is the metric that proves it.


In [13]:
def parametric_generate(query: str, retrieved_docs: list) -> str:
    '''Deliberately ignores retrieved_docs and answers from a fixed, hardcoded response --
    the bug: retrieval succeeded, generation didn't use it.'''
    return "That's a well-established fact from general background knowledge."


print("--- Bug: retrieval is correct, generation ignores it ---\n")
qa = qa_pairs[10]
retrieved_ids = tfidf_retriever.retrieve(qa["question"], k=3)
print(f"Question: {qa['question']}")
print(f"Retrieved: {retrieved_ids} -- correct: {qa['gold_doc_id'] in retrieved_ids}")

retrieved_docs = [doc_lookup[d] for d in retrieved_ids]
bad_answer = parametric_generate(qa["question"], retrieved_docs)
bad_faithfulness = eval_metrics.faithfulness_score(bad_answer, [d["text"] for d in retrieved_docs])
print(f"Answer: {bad_answer!r}")
print(f"Faithfulness score: {bad_faithfulness:.2f}  <-- low, despite retrieval being correct")


--- Bug: retrieval is correct, generation ignores it ---

Question: What cultures were part of Kublai's administration?
Retrieved: ['squad-005', 'squad-008', 'squad-007'] -- correct: True
Answer: "That's a well-established fact from general background knowledge."
Faithfulness score: 0.43  <-- low, despite retrieval being correct


In [14]:
print("--- Fix: generator actually uses the retrieved context ---\n")
good_answer = generate(qa["question"], retrieved_docs)
good_faithfulness = eval_metrics.faithfulness_score(good_answer, [d["text"] for d in retrieved_docs])
print(f"Answer: {good_answer!r}")
print(f"Faithfulness score: {good_faithfulness:.2f}")
print(f"\nSame retrieval both times ({retrieved_ids}) -- the only thing that changed is whether generation used it.")


--- Fix: generator actually uses the retrieved context ---

Answer: "Chinese advisers such as Liu Bingzhong and Yao Shu gave strong influence to Kublai's early court, and the central government administration was established within the first decade of Kublai's reign. Between 1402 and 1405, the expedition led by the Norman noble Jean de Bethencourt and the Poitevine Gadifer de la Salle conquered the Canarian islands of Lanzarote, Fuerteventura and El Hierro off the Atlantic coast of Africa. Retention rates for the final two years of secondary school were 77 per cent for public school students and 90 per cent for private school students."
Faithfulness score: 1.00

Same retrieval both times (['squad-005', 'squad-008', 'squad-007']) -- the only thing that changed is whether generation used it.


## Break it #3: the answer is split across two chunks

Real SQuAD answers are usually self-contained within one passage, so this needs one small,
clearly-labeled synthetic example layered on top (same allowance the spec gives the
confusable documents above): two short documents where neither alone answers the question,
only their combination does. Naive top-k retrieval with k set too low grabs only one.


In [15]:
SPLIT_FACT_DOCS = [
    {"doc_id": "split-fact-A", "title": "Project Kestrel (part 1)",
     "text": "Project Kestrel was founded in 2019 as an internal research initiative."},
    {"doc_id": "split-fact-B", "title": "Project Kestrel (part 2)",
     "text": "Project Kestrel's founding team consisted of Priya Nair and Tom Alcock."},
]
split_fact_lookup = {d["doc_id"]: d for d in SPLIT_FACT_DOCS}
split_fact_retriever = TfidfRetriever(SPLIT_FACT_DOCS)

split_query = "Who founded Project Kestrel and in what year?"

print("--- Bug: k=1 only retrieves one half of the answer ---\n")
top_1 = split_fact_retriever.retrieve(split_query, k=1)
print(f"Retrieved (k=1): {top_1}")
partial_context = [split_fact_lookup[d] for d in top_1]
partial_answer = generate(split_query, partial_context)
print(f"Answer: {partial_answer!r}")
print("(only has the year, or only the founders -- never both, since only one chunk was retrieved)")


--- Bug: k=1 only retrieves one half of the answer ---

Retrieved (k=1): ['split-fact-A']
Answer: 'Project Kestrel was founded in 2019 as an internal research initiative.'
(only has the year, or only the founders -- never both, since only one chunk was retrieved)


In [16]:
print("--- Fix: retrieve enough chunks (k=2) to cover both facts ---\n")
top_2 = split_fact_retriever.retrieve(split_query, k=2)
print(f"Retrieved (k=2): {top_2}")
full_context = [split_fact_lookup[d] for d in top_2]
full_answer = generate(split_query, full_context)
print(f"Answer: {full_answer!r}")
print("\nRaising k isn't a universal fix (it also raises noise and cost) -- the real lesson is")
print("that naive top-k retrieval has no way to know an answer needs multiple chunks at all;")
print("production systems handle this with larger k plus a reranker, or chunk-linking at index time.")


--- Fix: retrieve enough chunks (k=2) to cover both facts ---

Retrieved (k=2): ['split-fact-A', 'split-fact-B']
Answer: "Project Kestrel was founded in 2019 as an internal research initiative. Project Kestrel's founding team consisted of Priya Nair and Tom Alcock."

Raising k isn't a universal fix (it also raises noise and cost) -- the real lesson is
that naive top-k retrieval has no way to know an answer needs multiple chunks at all;
production systems handle this with larger k plus a reranker, or chunk-linking at index time.


## Break it #4: the near-duplicate confusable document wins

The confusable documents generated earlier are, by construction, nearly identical to a real
passage except for one swapped fact, similar enough that a retriever can rank the wrong one
first.

In [17]:
print("--- Bug: a query about the original fact retrieves the confusable instead ---\n")
confusable_hits = 0
for c in confusables:
    original = next(d for d in docs if d["doc_id"] == c["confusable_of"])
    matching_qas = [qa for qa in qa_pairs if qa["gold_doc_id"] == c["confusable_of"]]
    if not matching_qas:
        continue
    qa = matching_qas[0]
    top_1 = tfidf_retriever.retrieve(qa["question"], k=1)
    if top_1 == [c["doc_id"]]:
        confusable_hits += 1
        print(f"Question: {qa['question']}")
        print(f"Retrieved: {top_1} (the CONFUSABLE, not {c['confusable_of']!r})")
        print(f"Synthetic change in play: {c['synthetic_change']}")
        wrong_answer = generate(qa["question"], [doc_lookup[c["doc_id"]]])
        print(f"Answer (confidently uses the swapped fact): {wrong_answer!r}\n")

print(f"{confusable_hits} of {len(confusables)} confusable documents won top-1 over their original at least once.")


--- Bug: a query about the original fact retrieves the confusable instead ---

Question: Theories on imperialism use which country as a model?
Retrieved: ['squad-027-confusable'] (the CONFUSABLE, not 'squad-027')
Synthetic change in play: name 'Minister Benjamin' -> 'Sarah Thompson'
Answer (confidently uses the swapped fact): 'In anglophone academic works, theories regarding imperialism are often based on the British experience.'

Question: What is the Mongolian name of the first Mongolian laws codified in writing?
Retrieved: ['squad-003-confusable'] (the CONFUSABLE, not 'squad-003')
Synthetic change in play: name 'Genghis Khan' -> 'Sarah Thompson'
Answer (confidently uses the swapped fact): 'He is also given credit for the introduction of the traditional Mongolian script and the creation of the Ikh Zasag (Great Administration), the first written Mongolian law.'

Question: How common was the form of corporal punishment in the past?
Retrieved: ['squad-011-confusable'] (the CONFUSABLE, n

In [18]:
print("--- Fix: catch near-duplicates at index time instead of relying on retrieval to sort it out ---\n")
clean_docs = [d for d in all_docs if "confusable_of" not in d]
clean_retriever = TfidfRetriever(clean_docs)

for c in confusables[:1]:
    matching_qas = [qa for qa in qa_pairs if qa["gold_doc_id"] == c["confusable_of"]]
    if matching_qas:
        qa = matching_qas[0]
        top_1 = clean_retriever.retrieve(qa["question"], k=1)
        print(f"Question: {qa['question']}")
        print(f"Retrieved: {top_1} -- correct: {top_1 == [c['confusable_of']]}")

print("\nThe real fix for this class of bug is upstream of retrieval entirely: catch and merge")
print("or flag near-duplicates during ingestion (the deduplication step built earlier in this")
print("chapter), rather than hoping the retriever always breaks the tie correctly at query time.")


--- Fix: catch near-duplicates at index time instead of relying on retrieval to sort it out ---

Question: Theories on imperialism use which country as a model?
Retrieved: ['squad-027'] -- correct: True

The real fix for this class of bug is upstream of retrieval entirely: catch and merge
or flag near-duplicates during ingestion (the deduplication step built earlier in this
chapter), rather than hoping the retriever always breaks the tie correctly at query time.


## Break it #5: offline metrics improve, online satisfaction drops

A simulated case, clearly labeled as such: a retriever change that objectively improves the
offline eval set above, evaluated against a separate, held-out sample of more realistic
(conversational, imprecise) user queries. This is the offline-vs-online divergence every
production ML/RAG team eventually hits.


In [19]:
class BigramTfidfRetriever:
    '''v2: adds bigrams (2-word phrases) to the vectorizer on top of v1's unigrams-only
    TF-IDF. This rewards exact phrase matches more heavily -- a real, mechanistic change,
    not just a relabeling -- which tends to help on precisely-phrased benchmark questions
    and tends to hurt on loosely-phrased, conversational real queries where exact 2-word
    phrase overlap with the source passage is much rarer.'''
    def __init__(self, docs: list):
        self.doc_ids = [d["doc_id"] for d in docs]
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.doc_vectors = self.vectorizer.fit_transform([d["text"] for d in docs])

    def retrieve(self, query: str, k: int = 3) -> list:
        query_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(query_vec, self.doc_vectors)[0]
        ranked = sims.argsort()[::-1][:k]
        return [self.doc_ids[i] for i in ranked]


v1_retriever = tfidf_retriever  # unigrams, already built above
v2_retriever = BigramTfidfRetriever(all_docs)

v1_offline = eval_metrics.evaluate_retrieval(eval_queries, v1_retriever.retrieve, k=3, **metric_fns)
v2_offline = eval_metrics.evaluate_retrieval(eval_queries, v2_retriever.retrieve, k=3, **metric_fns)

print("Offline eval (real, precisely-phrased SQuAD questions):")
print(f"  v1 (unigrams) recall@k: {v1_offline['recall@k']:.3f}")
print(f"  v2 (+ bigrams) recall@k: {v2_offline['recall@k']:.3f}")


Offline eval (real, precisely-phrased SQuAD questions):
  v1 (unigrams) recall@k: 0.945
  v2 (+ bigrams) recall@k: 0.964


In [20]:
REALISTIC_QUERIES = [
    "hey quick question, who started anthropic again?",
    "can you remind me what react stands for in the agent context",
    "what happened with genghis khan and the shah's army",
    "kublai khan's advisors -- who were they",
    "corporal punishment in schools -- how common was it historically",
]


def simulated_satisfaction(retriever_name: str, query: str) -> float:
    '''Deliberately hand-constructed, deterministic stand-in for a real user-satisfaction
    signal (e.g. thumbs-up rate on live traffic) -- this is NOT derived from the retrievers'
    actual behavior above, and it is not meant to be: real online/offline divergence isn't
    always mechanistically traceable to one code change the way this notebook's other
    break-it bugs are, which is exactly what makes it dangerous in practice -- a team ships
    v2 because recall@k went up, and the online signal (which most teams check far less
    often than an offline eval suite) quietly moves the other way for unrelated or
    hard-to-pin-down reasons. Modeled here as v1 scoring higher than v2 on average, fixed
    seed per query for reproducibility.'''
    rng = random.Random(hash((retriever_name, query)) % (2**32))
    base = 0.78 if retriever_name == "v1" else 0.58
    return max(0.0, min(1.0, base + rng.uniform(-0.07, 0.07)))


v1_satisfaction = [simulated_satisfaction("v1", q) for q in REALISTIC_QUERIES]
v2_satisfaction = [simulated_satisfaction("v2", q) for q in REALISTIC_QUERIES]

print("Simulated online satisfaction (held-out, realistic/conversational queries):")
print(f"  v1 (unigrams) avg satisfaction:  {sum(v1_satisfaction) / len(v1_satisfaction):.3f}")
print(f"  v2 (+ bigrams) avg satisfaction: {sum(v2_satisfaction) / len(v2_satisfaction):.3f}")
print()
print("v2 won the offline benchmark above (higher recall@k) and loses here. The offline eval")
print("set's precisely-phrased benchmark questions and real users' loose, conversational")
print("phrasing are not the same distribution -- a change that helps on one can hurt on the")
print("other, and it doesn't always show up as a clean mechanistic story you can point to.")
print("Shipping on the strength of the offline number alone, without ever checking a held-out")
print("online signal, is the actual mistake this scenario is about.")


Simulated online satisfaction (held-out, realistic/conversational queries):
  v1 (unigrams) avg satisfaction:  0.801
  v2 (+ bigrams) avg satisfaction: 0.571

v2 won the offline benchmark above (higher recall@k) and loses here. The offline eval
set's precisely-phrased benchmark questions and real users' loose, conversational
phrasing are not the same distribution -- a change that helps on one can hurt on the
other, and it doesn't always show up as a clean mechanistic story you can point to.
Shipping on the strength of the offline number alone, without ever checking a held-out
online signal, is the actual mistake this scenario is about.


## Interview preparation

### Recap

- Retrieval, ranking, and generation are three separate failure surfaces. Precision@k and
  recall@k diagnose retrieval, MRR diagnoses ranking, and faithfulness diagnoses generation.
  Diagnosing which one failed from a symptom description is a real, common interview
  question.
- RAG reduces hallucination risk; it does not eliminate it. This chapter demonstrated why
  hands-on, not just asserted it (break-it #2).
- Chunking strategy and near-duplicate detection are ingestion-time decisions that determine
  what retrieval can even find. "How would you chunk a messy corpus" is a genuine, commonly
  asked question, answerable now from having actually built both a fixed-size and a semantic
  chunker against the same real messy document.
- Offline metrics and real user satisfaction can diverge (break-it #5). An improved
  benchmark score is not proof a change should ship.

### Cold-diagnosis exercise

For each scenario, decide: is this a retrieval problem or a generation problem, and which
metric proves it? Attempt from memory, then check `solutions/ch03_rag_evaluation_answers.md`.

1. A RAG system gives a confident, detailed, entirely wrong answer. Offline precision@k on
   this query is 0.
2. A RAG system gives a confident, fluent answer that doesn't match the source material at
   all, but precision@k and recall@k on this query are both 1.0.
3. A RAG system's answer is half-right: it has the entity but not the date, or vice versa.
4. A RAG system gives a plausible answer that's subtly wrong in one specific fact (a date, a
   name), and the source documents contain a near-identical passage with a different value
   for that exact fact.

### Further exercises

5. How would you chunk a messy real-world corpus, and how would you catch near-duplicate
   chunks before they hurt retrieval? (Answer from what you actually built in this chapter's
   ingestion section, not from general principles alone.)
6. Write, in plain language suitable for a non-technical CEO, why an AI system that uses RAG
   can still hallucinate. No jargon (no "retrieval," "embeddings," "faithfulness").

Check your answers against `solutions/ch03_rag_evaluation_answers.md`.


In [21]:
my_ceo_explanation = '''
(Write your answer to question 6 here.)
'''

print(my_ceo_explanation)



(Write your answer to question 6 here.)



## Further practice (optional)

SQuAD 1.1 alone is sufficient to complete this chapter, but two free, well-documented
corpora are natural next steps if you want more practice in a different domain. Neither was
runnable in this build environment (both are Hugging-Face-hosted, and this repo's build
environment can't reach `huggingface.co`; see `PROGRESS.md`'s Unit 4 notes), so they're
pointers, not executed cells here.

PubMedQA (`qiaojin/PubMedQA`, MIT license) is a domain-specific medical RAG variant with
yes/no/maybe plus long-form answers grounded in real PubMed abstracts. It's good for seeing
how retrieval behaves in a domain with denser, more technical vocabulary than Wikipedia.
BEIR, specifically the FiQA subset, is a standardized heterogeneous retrieval benchmark,
useful for seeing how precision@k/recall@k/MRR behave across a genuinely different domain
(finance Q&A) than this chapter's Wikipedia-derived passages.

## Next: Chapter 4: Production Reliability

This chapter treated retrieval and generation as if they always run correctly when called.
Chapter 4 is about what happens when they don't: stale caches, flaky tools, and cascading
failures, the gap between a demo and something that survives production traffic.
